# Attention Learns the Alignment You Never Labelled

**Topic:** Encoder-decoder attention &nbsp;·&nbsp; **Difficulty:** Medium

Before attention, a seq2seq model squeezed the entire source sentence into one
vector and the decoder had to work from that. Attention lets each output step
look back at every input position, and the striking part is that the alignment it
learns (which source word each target word attends to) was never supervised.

Fit a tiny additive-attention alignment on a copy-with-offset task and check the
attention matrix against the true alignment.

## The data (created in the next cell)

- `enc` `(14, 16)`: encoder states for a 14-token source.
- `dec` `(14, 16)`: decoder states, built so target position `t` should
  attend to source position `t - 2` (wrapping at the start).
- `true_align` `(14,)`: the true alignment, used as the training target and
  for scoring (never to build the model).

## Your task

With **NumPy only**:

1. Implement additive (Bahdanau) attention:
   `score(h_i, s_t) = v^T tanh(W_e h_i + W_d s_t)`, softmax over `i`.
2. Train `v`, `W_e`, `W_d` by gradient descent for 3000 steps at lr 0.05 to
   maximize the log-probability the attention assigns to the true alignment
   (cross-entropy on the attention row against `true_align`), seeded with
   `np.random.default_rng(44)`.
3. Set `answers` to a dict with `"final_loss"`, `"argmax_accuracy"` (fraction of
   target positions whose attention argmax equals the true source index),
   `"mean_true_prob"` (mean attention mass on the correct source position), and
   `"attn_entropy"` (mean entropy in bits of the attention rows).

## Evaluation

Compared to the reference to a relative tolerance of 1e-5. The evaluation cell
checks the learned alignment is at least 90% correct and the attention is sharp.

In [ ]:
import numpy as np

rng = np.random.default_rng(4004)
T, d = 14, 16
enc = rng.normal(0, 1, (T, d))
true_align = (np.arange(T) - 2) % T
dec = enc[true_align] * 0.9 + rng.normal(0, 0.25, (T, d))
print(f"enc {enc.shape}, target position t should attend to source (t-2) mod {T}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("final_loss", [], 0.0032456498837018445, -0.0005434451733099469, 1e-05),
    ("argmax_accuracy", [], 1.0, -0.16743801481449916, 1e-05),
    ("mean_true_prob", [], 0.9967600977140056, -0.16689553200753932, 1e-05),
    ("attn_entropy", [], 0.040739502004159336, -0.006821341340107749, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "argmax_accuracy" in answers:
    _acc = answers["argmax_accuracy"] >= 0.9
    _sharp = answers["attn_entropy"] < 1.5
    print(f"  {'alignment recovered':<24} {'ok' if _acc else 'wrong'}  "
          f"({answers['argmax_accuracy']:.3f} argmax accuracy)")
    print(f"  {'attention is sharp':<24} {'ok' if _sharp else 'wrong'}  "
          f"({answers['attn_entropy']:.3f} bits)")
    _ok = _ok and _acc and _sharp

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The model has no access to `true_align` except through the loss, and the score
function is a generic bilinear-ish form, and nothing about it encodes "look two back".
It finds the offset because the decoder states genuinely carry a signature of the
source state they came from, and attention's job is precisely to surface that.
The low entropy at the end is the model committing: an attention row that stays
near-uniform is a model that has learned nothing to align.

```python
import numpy as np

rs = np.random.default_rng(44)
We = rs.normal(0, 0.3, (d, d))
Wd = rs.normal(0, 0.3, (d, d))
v = rs.normal(0, 0.3, d)

onehot = np.zeros((T, T))
onehot[np.arange(T), true_align] = 1.0

lr = 0.05
for _ in range(3000):
    A = enc @ We.T                       # (T_src, d)
    B = dec @ Wd.T                       # (T_tgt, d)
    Z = np.tanh(A[None, :, :] + B[:, None, :])       # (T_tgt, T_src, d)
    S = Z @ v                                          # (T_tgt, T_src)
    S = S - S.max(axis=1, keepdims=True)
    E = np.exp(S)
    P = E / E.sum(axis=1, keepdims=True)

    dS = (P - onehot) / T
    gv = np.einsum("ts,tsd->d", dS, Z)
    dZ = dS[:, :, None] * v[None, None, :] * (1 - Z ** 2)
    gWe = np.einsum("tsd,se->de", dZ, enc)
    gWd = np.einsum("tsd,te->de", dZ, dec)

    v -= lr * gv
    We -= lr * gWe
    Wd -= lr * gWd

loss = float(-np.log(P[np.arange(T), true_align] + 1e-12).mean())
ent = float(-(P * np.log2(np.where(P > 0, P, 1.0))).sum(axis=1).mean())

answers = {
    "final_loss": loss,
    "argmax_accuracy": float((P.argmax(axis=1) == true_align).mean()),
    "mean_true_prob": float(P[np.arange(T), true_align].mean()),
    "attn_entropy": ent,
}
```